In [ ]:
from typing import TypedDict, List
import random, string
from langgraph.graph import StateGraph, START, END

In [ ]:
class AgentState(TypedDict):
    draft: str
    feedback: List[str]
    attempt: int
    max_attempts: int
    accepted: bool

In [ ]:
def generate(state: AgentState) -> AgentState:
    state["attempt"] += 1
    pool = string.ascii_lowercase
    if "needs uppercase" in state["feedback"]:
        pool += string.ascii_uppercase
    if "needs digit" in state["feedback"]:
        pool += string.digits
    if "needs symbol" in state["feedback"]:
        pool += "!@#$%&*"
    length = 6 if "too short" not in state["feedback"] else 12
    state["draft"] = "".join(random.choice(pool) for _ in range(length))
    print(f"[attempt {state['attempt']}] generate -> {state['draft']}")
    return state

In [ ]:
def critic(state: AgentState) -> AgentState:
    pw = state["draft"]
    issues = []
    if len(pw) < 10:
        issues.append("too short")
    if not any(c.isupper() for c in pw):
        issues.append("needs uppercase")
    if not any(c.isdigit() for c in pw):
        issues.append("needs digit")
    if not any(c in "!@#$%&*" for c in pw):
        issues.append("needs symbol")
    state["feedback"] = issues
    state["accepted"] = len(issues) == 0
    print(f"            critic   -> {'accept' if state['accepted'] else issues}")
    return state

In [ ]:
def decide(state: AgentState) -> str:
    if state["accepted"]:
        return "done"
    if state["attempt"] >= state["max_attempts"]:
        return "giveup"
    return "retry"

In [ ]:
graph = StateGraph(AgentState)

In [ ]:
graph.add_node("generate", generate)
graph.add_node("critic", critic)

graph.set_entry_point("generate")
graph.add_edge("generate", "critic")

graph.add_conditional_edges(
    "critic",
    decide,
    {"retry": "generate", "done": END, "giveup": END},
)

In [ ]:
test = graph.compile()

In [ ]:
from IPython.display import Image, display
display(Image(test.get_graph().draw_mermaid_png()))

In [ ]:
random.seed(7)
result = test.invoke({
    "draft": "",
    "feedback": [],
    "attempt": 0,
    "max_attempts": 6,
    "accepted": False,
})

In [ ]:
result